Install the require Libraries

In [1]:
!pip -q install pinecone-client
!pip -q install langchain_openai
!pip -q install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00


Initialize the APIs

In [56]:
import os
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import Pinecone
from langchain_community.chat_models import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage
from pinecone import Pinecone
from google.colab import userdata
# Set API Keys
os.environ["OPENAI_API_KEY"] = userdata.get('openai_api_key')
os.environ["PINECONE_API_KEY"] = userdata.get('pinecone_api_key')


# Initialize Pinecone Client
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# Connect to existing index
index_name = "medical-paitents-dataset"
index = pc.Index(index_name)

# Initialize LangChain Pinecone Vector Store
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

vector_db = Pinecone(
    index,  # Replace with your actual index name
    embedding=embedding_model,
    text_key="Symptoms"
)
# Initialize Chat Model (LLM)
llm = ChatOpenAI(model_name="gpt-4o-mini")



This Function is for Vector Search in Pinecone with threshold.
Right now Threshold is set to 0 for testing purposes.

In [57]:
def search_similar_cases(query, threshold=0):
    """Search Pinecone for similar medical cases with a score threshold"""
    # Connect to existing index
    index_name = "medical-paitents-dataset"
    index = pc.Index(index_name)
    # Generate embedding for the query
    query_embedding = embedding_model.embed_query(query)

    # Query Pinecone for top-k matches
    results = index.query(
        vector=query_embedding,
        top_k=5,  # Fetch more results, then filter
        include_metadata=True,
        include_values=True
    )

    print("🔎 Similar Cases (Above 80% Similarity):")

    similar_cases = [] # Initialize an empty list to store similar cases

    for res in results['matches']:
        score = res['score']  # Get similarity score
        text = res['metadata'].get('text', 'No metadata found')  # Handle missing metadata safely

        # Convert score to percentage & apply threshold
        similarity_percentage = score * 100  # Convert to percentage
        if similarity_percentage >= (threshold * 100):  # 0% similarity check
            print(f"- {text} (Score: {similarity_percentage:.2f}%)")
            similar_cases.append(text) # Append the similar case text to the list

    return similar_cases # Return the list of similar cases
search_similar_cases("I have severe headache, bad throat and fever")

🔎 Similar Cases (Above 80% Similarity):
- Patient ID: 19 Name: Mia Smith Age: 24 Gender: Female Symptoms: Severe sore throat, fever Medical History: nan Possible Diagnoses: Strep throat Medications: Arinac, Hydryllin syrup (Score: 49.60%)
- Patient ID: 47 Name: Lily Taylor Age: 26 Gender: Female Symptoms: Severe sore throat, fever Medical History: Strep throat Possible Diagnoses: Strep throat Medications: Arinac, Hydryllin syrup (Score: 48.31%)
- Patient ID: 7 Name: Frank Miller Age: 40 Gender: Male Symptoms: Sore throat, fever, swollen glands Medical History: nan Possible Diagnoses: Strep throat, Tonsillitis Medications: Arinac, Hydryllin syrup (Score: 47.80%)
- Patient ID: 71 Name: Scarlett Clark Age: 6 Gender: Female Symptoms: Severe sore throat, fever Medical History: Strep throat Possible Diagnoses: Strep throat Medications: Arinac syrup (Score: 46.54%)
- Patient ID: 95 Name: Scarlett Taylor Age: 6 Gender: Female Symptoms: Severe sore throat, fever Medical History: Strep throat Po

['Patient ID: 19 Name: Mia Smith Age: 24 Gender: Female Symptoms: Severe sore throat, fever Medical History: nan Possible Diagnoses: Strep throat Medications: Arinac, Hydryllin syrup',
 'Patient ID: 47 Name: Lily Taylor Age: 26 Gender: Female Symptoms: Severe sore throat, fever Medical History: Strep throat Possible Diagnoses: Strep throat Medications: Arinac, Hydryllin syrup',
 'Patient ID: 7 Name: Frank Miller Age: 40 Gender: Male Symptoms: Sore throat, fever, swollen glands Medical History: nan Possible Diagnoses: Strep throat, Tonsillitis Medications: Arinac, Hydryllin syrup',
 'Patient ID: 71 Name: Scarlett Clark Age: 6 Gender: Female Symptoms: Severe sore throat, fever Medical History: Strep throat Possible Diagnoses: Strep throat Medications: Arinac syrup',
 'Patient ID: 95 Name: Scarlett Taylor Age: 6 Gender: Female Symptoms: Severe sore throat, fever Medical History: Strep throat Possible Diagnoses: Strep throat Medications: Arinac syrup']

TO get the response from the LLM we use this function.

In [58]:
def chat_with_llm(user_query):
    """Retrieve context from Pinecone and chat with LLM"""
    retrieved_cases = search_similar_cases(user_query)

    # Combine retrieved cases as context
    context = "\n".join(retrieved_cases)  # Directly join the retrieved cases

    # Define messages for LLM
    messages = [
        SystemMessage(content="You are an AI medical assistant. Provide diagnosis based on retrieved cases."),
        HumanMessage(content=f"Patient Symptoms: {user_query}\n\nRelevant Cases:\n{context}\n\nWhat is the possible diagnosis?")
    ]

    # Get response from GPT-4o-mini
    response = llm(messages)
    return response.content


In [59]:
# a sample qurey test.
user_input = "I have severe headache and dizziness"
response = chat_with_llm(user_input)
print("\n💬 AI Response:", response)


🔎 Similar Cases (Above 80% Similarity):
- Patient ID: 21 Name: Amelia Davis Age: 33 Gender: Female Symptoms: Severe headache, nausea Medical History: Migraine Possible Diagnoses: Migraine Medications: Ponstan forte (Score: 50.22%)
- Patient ID: 65 Name: Hannah Clark Age: 6 Gender: Female Symptoms: Severe headache, nausea Medical History: Migraine Possible Diagnoses: Migraine Medications: Panadol syrup (Score: 50.01%)
- Patient ID: 49 Name: Hannah Wilson Age: 37 Gender: Female Symptoms: Severe headache, nausea Medical History: Migraine Possible Diagnoses: Migraine Medications: Ponstan forte (Score: 49.84%)
- Patient ID: 2 Name: Jane Smith Age: 28 Gender: Female Symptoms: Headache, nausea, dizziness Medical History: Migraine history Possible Diagnoses: Migraine, Dehydration Medications: Ponstan forte, Motilium (Score: 49.68%)
- Patient ID: 54 Name: Daniel Clark Age: 7 Gender: Male Symptoms: Headache, dizziness Medical History: nan Possible Diagnoses: Migraine Medications: Panadol syrup (